In [0]:
%python
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
%python
spark.sql("SHOW SCHEMAS").show(truncate=False)

In [0]:
%python
spark.sql("""
SELECT current_catalog(), current_schema()
""").show()

In [0]:
%python
spark.sql("""
SHOW TABLES IN workspace.lab_db
""").show(truncate=False)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.lab_db.lab_landing;

In [0]:
landing_path = "/Volumes/workspace/lab_db/lab_landing/orders/"
checkpoint_path = "/Volumes/workspace/lab_db/lab_landing/checkpoints/orders_autoloader/"

print("Landing path:", landing_path)
print("Checkpoint path:", checkpoint_path)

In [0]:
dbutils.fs.mkdirs(landing_path)
dbutils.fs.mkdirs(checkpoint_path)

print("Directories created successfully")

In [0]:
display(dbutils.fs.ls(landing_path))

In [0]:
orders = [
    {
        "order_id": "ORD001",
        "customer_id": 101,
        "amount": 250.50,
        "order_timestamp": "2026-09-29T10:00:00"
    },
    {
        "order_id": "ORD002",
        "customer_id": 102,
        "amount": 480.00,
        "order_timestamp": "2026-09-29T10:05:00"
    },
    {
        "order_id": "ORD003",
        "customer_id": 103,
        "amount": 125.75,
        "order_timestamp": "2026-09-29T10:10:00"
    }
]

import json

for order in orders:
    file_path = landing_path + order["order_id"] + ".json"
    dbutils.fs.put(
        file_path,
        json.dumps(order),
        overwrite=True
    )

print("3 JSON order files created")

In [0]:
display(dbutils.fs.ls(landing_path))

In [0]:
print(dbutils.fs.head(landing_path + "ORD001.json"))

In [0]:
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load(landing_path)
)

In [0]:
(
    df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path + "write/")
    .trigger(availableNow=True)
    .table("workspace.lab_db.bronze_orders")
)

In [0]:
spark.sql("""
SELECT *
FROM workspace.lab_db.bronze_orders
ORDER BY order_id
""").show(truncate=False)